# exp005 — checkpoint variance: 6 ink_9um checkpoints x 3 conditions (Kaggle)

**Settings:** Accelerator **GPU T4 x2** (1 GPU used), Internet **on** (turn it back on after enabling the GPU), notebook **private**.
Config: `configs/exp005_checkpoints_ink9um.yaml`. Downloads into `/kaggle/working` (discarded with the session):
6 checkpoints x 138 MB (~830 MB) + data ~1.15 GB (841 window ~275 MB, Paris 4 window ~114 MB, 3 validation regions ~757 MB).
Numbers only, no images. Paris 4 = Scroll 1 (EduceLab terms, no academic claims).

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
python --version; df -h /kaggle/working | tail -1
git clone -q --depth 1 https://github.com/andreluizpedroso/Vesuvius-Challenge.git /kaggle/working/harness
git -C /kaggle/working/harness log -1 --format='harness %H'

In [ ]:
import os, yaml, pathlib
H = pathlib.Path("/kaggle/working/harness")
EXP = "configs/exp005_checkpoints_ink9um.yaml"
cfg = yaml.safe_load((H / EXP).read_text())
val_cfg = yaml.safe_load((H / cfg["conditions"]["valregions"]["segments_from"]).read_text())
c = cfg["conditions"]
os.environ.update({
    "H": str(H), "EXP_CFG": str(H / EXP), "EXP_NAME": cfg["experiment"],
    "VILLA_SHA": cfg["upstream"]["villa_commit"], "PY_VER": cfg["upstream"]["python"],
    "UV_SYNC_ARGS": " ".join(cfg["upstream"]["uv_sync_args"]),
    "CK_REPO": cfg["checkpoint_repo"]["hf_repo"], "CK_REV": cfg["checkpoint_repo"]["hf_revision"],
    "CKPTS": " ".join(f"{k['id']}={k['file']}" for k in cfg["checkpoint_repo"]["checkpoints"]),
    "SEGS": " ".join(s["key"] for s in val_cfg["data"]["segments"]),
    "P_PARIS": f"{c['paris4']['prepare']['script']}|{c['paris4']['prepare']['config']}",
    "P_VAL": f"{c['valregions']['prepare']['script']}|{c['valregions']['prepare']['config']}",
    "P_841": f"{c['e841']['prepare']['script']}|{c['e841']['prepare']['config']}",
    "INFER_MODULE": cfg["inference"]["module"], "INFER_ARGS": " ".join(cfg["inference"]["base_args"]),
    "VES": "/kaggle/working/villa/vesuvius", "CK_DIR": "/kaggle/working/checkpoints/ink_9um",
    "DATA": "/kaggle/working/data", "OUT": f"/kaggle/working/outputs/{cfg['experiment']}",
})
print({k: os.environ[k] for k in ["EXP_NAME", "VILLA_SHA", "CKPTS", "SEGS"]})

## 1. Upstream code (villa, pinned commit) + environment

In [ ]:
%%bash
set -e
git init -q /kaggle/working/villa && cd /kaggle/working/villa
git remote add origin https://github.com/ScrollPrize/villa.git
git sparse-checkout set vesuvius volume-cartographer segmentation/models/batchgeneratorsv2 segmentation/models/arch/nnunet
git fetch -q --depth 1 --filter=blob:none origin "$VILLA_SHA"
git checkout -q FETCH_HEAD && git log -1 --format='villa %H %cd'

In [ ]:
%%bash
set -e
pip install -q uv==0.11.24
uv python install "$PY_VER"
cd "$VES" && uv sync --python "$PY_VER" $UV_SYNC_ARGS 2>&1 | tail -5
uv run --no-sync python -c "import torch, sys; print(sys.version.split()[0], torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 2. Data terms
Already accepted by the owner for this account in exp000. Nothing here runs `vesuvius.accept_terms`.
If a later cell fails with an error about terms/agreement, **stop and ask the owner**.

## 3. Checkpoints (pinned HF revision, 6 x 138 MB)

In [ ]:
%%bash
set -eo pipefail
for pair in $CKPTS; do
  f="${pair#*=}"
  uvx --from huggingface_hub==1.22.0 hf download "$CK_REPO" "$f" --revision "$CK_REV" --local-dir "$CK_DIR" 2>&1 | tail -1
done
find "$CK_DIR" -name "*.pth" | sort | xargs -I{} sh -c 'echo "$(stat -c %s {}) {}"'

## 4. Prepare the three conditions with the ORIGINAL scripts/configs (each aborts on its own sanity checks)

In [ ]:
%%bash
set -eo pipefail
IFS='|' read -r S C <<< "$P_PARIS"
cd "$VES" && uv run --no-sync python "$H/$S" --config "$H/$C" --out "$DATA/paris4" 2>&1 | grep -v -E "Unclosed|ResponseHandler|TCPConnector|ClientSession|^\s*$" | tail -45

In [ ]:
%%bash
set -eo pipefail
IFS='|' read -r S C <<< "$P_VAL"
cd "$VES" && uv run --no-sync python "$H/$S" --config "$H/$C" --out "$DATA/valregions" 2>&1 | grep -v -E "Unclosed|ResponseHandler|TCPConnector|ClientSession|^\s*$" | tail -120

In [ ]:
%%bash
set -eo pipefail
IFS='|' read -r S C <<< "$P_841"
cd "$VES" && uv run --no-sync python "$H/$S" --config "$H/$C" --out "$DATA/e841" 2>&1 | grep -v -E "Unclosed|ResponseHandler|TCPConnector|ClientSession|^\s*$" | tail -60

## 5. Inference: 6 checkpoints x (Paris 4 fwd, 3 validation regions fwd, 841 fwd+rev) at the default depth window

In [ ]:
%%bash
set -eo pipefail
cd "$VES"
infer() { uv run --no-sync python -m "$INFER_MODULE" "$1" "$2" "$3" $INFER_ARGS --direction "$4" 2>&1 | grep -E "Error|Traceback|Selected source layer" | head -3; }
for pair in $CKPTS; do
  id="${pair%%=*}"; ck="$CK_DIR/${pair#*=}"
  echo "=== $id $(date +%T) ==="
  mkdir -p "$OUT/$id/paris4" "$OUT/$id/e841"
  infer "$DATA/paris4/input.zarr" "$ck" "$OUT/$id/paris4/pred_off0.tif" forward
  for seg in $SEGS; do
    mkdir -p "$OUT/$id/valregions/$seg"
    infer "$DATA/valregions/$seg/input.zarr" "$ck" "$OUT/$id/valregions/$seg/pred_off0.tif" forward
  done
  infer "$DATA/e841/input.zarr" "$ck" "$OUT/$id/e841/pred_off0.tif" both
done
echo "tifs: $(find "$OUT" -name '*.tif' | wc -l) (expected 36)"

## 6. Metrics (numbers only)

In [ ]:
%%bash
set -e
cd "$VES" && uv run --no-sync python "$H/scripts/exp005_eval.py" --config "$EXP_CFG" --data "$DATA" --preds "$OUT" --out "$OUT/metrics" --repo-root "$H"
for n in paris4 valregions e841; do cp "$DATA/$n/prep.json" "$OUT/metrics/prep_$n.json"; done
ls -la "$OUT/metrics"

## 7. Bring results back
Files in `/kaggle/working/outputs/exp005_checkpoints_ink9um/metrics/`: `metrics.json`, `report.md`, `prep_paris4.json`,
`prep_valregions.json`, `prep_e841.json`. Verify each against `sha256sum` / `wc -c` run in a cell. Do **not** download the TIFFs.